In [1]:
!pip install -q python-dotenv trainer datasets transformers huggingface_hub fsspec

ERROR: Could not find a version that satisfies the requirement trainer (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: No matching distribution found for trainer


In [2]:
!pip install -U -q datasets


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import torch
import os
from dotenv import load_dotenv

import transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments

load_dotenv(".env")

True

In [3]:
# 1. Load a suitable dataset (using a small subset for faster training)
# We'll use a subset of the SQuAD dataset
dataset = load_dataset("squad")

# Shuffle the dataset splits
dataset = dataset.shuffle(seed=42)

# Select a subset of examples from the train and validation splits
# Adjust the range as needed for the desired number of examples in each split
dataset["train"] = dataset["train"].select(range(800))  # Example: select 800 training examples
dataset["validation"] = dataset["validation"].select(range(200)) # Example: select 200 validation examples

In [4]:
# 2. Load a pre-trained Q&A model and its tokenizer
model_name = "distilbert-base-uncased" # Using DistilBERT for faster training
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# 3. Preprocess the dataset
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        start_char = answers[i]["answer_start"][0]
        end_char = start_char + len(answers[i]["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [6]:
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [10]:
# 4. Define training arguments (adjust for shorter training)
training_args = TrainingArguments(
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # Train for only 1 epoch
    weight_decay=0.01,
)

In [11]:
# 5. Create a Trainer and start training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
)

trainer.train()

<ipython-input-11-5db370d7b6f8>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Define a question and context
question = "What is the capital of France?"
context = "Paris is the capital of France. It is known for its art, fashion, and cuisine."

# Tokenize the input
inputs = tokenizer(question, context, return_tensors="pt")

# Get model predictions
with torch.no_grad():
    outputs = model(**inputs)

# Find the answer span
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

# Get the answer tokens
predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]

# Decode the answer tokens to get the answer text
answer = tokenizer.decode(predict_answer_tokens)

print(f"Question: {question}")
print(f"Context: {context}")
print(f"Answer: {answer}")